[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/kgrid-objects/FAIR-DO-Workshop/HEAD?urlpath=lab/tree/HBOT3-KA/auxiliary/aux-notebook/hbot_treatment_target_ka_binder.ipynb%3Fkernel_name%3Djavascript)

# HBOT Treatment Target Knowledge Assembly on Binder

This notebook runs the real, governed KA implementation from `src/orchestrator.js` directly -- no reimplemented logic. The Burden KO's `ajv`/`ajv-formats` dependency is not preinstalled in this Binder image, so the first cell installs it at runtime with `npm install` before requiring the orchestrator.

`executeKnowledgeAssembly(request)` is called with no `wagnerAskYesNo`/`burdenAskQuestion` overrides, so the Wagner and Burden KOs fall back to their own default interactive prompters, which use this Jupyter kernel's native `globalThis.$$.input` mechanism to ask each question live -- no answers are hardcoded.

Run the cells in order.

In [ ]:
const { execSync } = require('node:child_process');

// Installs the real Burden KO's own declared dependencies (ajv, ajv-formats)
// at runtime, since they are not preinstalled in this Binder image.
execSync('npm install', { cwd: '../../../collection/HBOT-Regimen-Burden-KO', stdio: 'inherit' });

const { executeKnowledgeAssembly } = require('../../src/orchestrator');
console.log('HBOT Treatment Target KA orchestrator loaded.');

In [ ]:
// Only subject binding, HBOT case assertions, and the Margolis first-visit
// assessment are fixed sample data here: neither the HBOT Decision KO nor
// the Margolis KO owns an interactive collection capability, so the KA
// never derives these from a questionnaire either (Section 2.5/2.6).
const request = {
  request_id: 'req-binder-demo',
  requested_at: '2026-09-24T12:00:00Z',
  index_time: '2026-09-24T10:00:00Z',
  subject_binding: {
    subject_identifier: { system: 'https://example.org/mrn', value: 'MRN-001' },
    ulcer_identifier: { system: 'https://example.org/ulcer', value: 'ULCER-001' },
    care_episode_identifier: { system: 'https://example.org/episode', value: 'EPISODE-001' },
    source_evidence: {}
  },
  hbot_case_assertions: {
    dfu_confirmed: { value: true, source_evidence: {} },
    acute_surgical_intervention: { value: true, source_evidence: {} },
    not_healed_after_30_days: { value: false, source_evidence: {} }
  },
  margolis_first_visit_assessment: {
    wound_area: { value: 1, ucum_code: 'cm2' },
    wound_duration: { value: 4, ucum_code: 'wk' },
    first_visit_at: '2026-09-10T09:00:00Z',
    first_visit_attested: true
  }
};

// IJavascript's kernel evaluates each cell via vm.Script, which does not
// support top-level await, so the async call is wrapped in an IIFE instead.
(async () => {
  const result = await executeKnowledgeAssembly(request);
  console.log(JSON.stringify(result, null, 2));
})();